In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [2]:
df=sns.load_dataset('tips')
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [3]:
# predict what is the time of anyone's visiting
# target >> time

In [4]:
df.time.value_counts()

time
Dinner    176
Lunch      68
Name: count, dtype: int64

In [5]:
# EDA >> subjective
# encoding, missing value treatment, scaling >> automate

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [7]:
df.isna().sum()

total_bill    0
tip           0
sex           0
smoker        0
day           0
time          0
size          0
dtype: int64

In [8]:
# since time is nominal variable, we will use label encoding

from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
df.time=encoder.fit_transform(df.time)

In [9]:
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,0,2
1,10.34,1.66,Male,No,Sun,0,3
2,21.01,3.50,Male,No,Sun,0,3
3,23.68,3.31,Male,No,Sun,0,2
4,24.59,3.61,Female,No,Sun,0,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,0,3
240,27.18,2.00,Female,Yes,Sat,0,2
241,22.67,2.00,Male,Yes,Sat,0,2
242,17.82,1.75,Male,No,Sat,0,2


In [10]:
# dinner = 0, lunch =1
df.time.unique()

array([0, 1])

In [11]:
X=df.drop('time', axis=1)
y=df.time

In [12]:
X

,total_bill,tip,sex,smoker,day,size
0,16.99,1.01,Female,No,Sun,2
1,10.34,1.66,Male,No,Sun,3
2,21.01,3.50,Male,No,Sun,3
3,23.68,3.31,Male,No,Sun,2
4,24.59,3.61,Female,No,Sun,4
...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,3
240,27.18,2.00,Female,Yes,Sat,2
241,22.67,2.00,Male,Yes,Sat,2
242,17.82,1.75,Male,No,Sat,2


In [13]:
y

0      0
1      0
2      0
3      0
4      0
      ..
239    0
240    0
241    0
242    0
243    0
Name: time, Length: 244, dtype: int64

In [14]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=1)

In [15]:
# handling the missing value
from sklearn.impute import SimpleImputer #automatic handels missing value
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.pipeline import Pipeline
# a sequence of data transformer
from sklearn.compose import ColumnTransformer #group all the pipiline steps for each columns


In [16]:
df.sample(1)

,total_bill,tip,sex,smoker,day,time,size
131,20.27,2.83,Female,No,Thur,1,2


In [17]:
cat_col=['sex','smoker','day']
num_col=['total_bill', 'tip', 'size']

In [18]:
# feature eng automation
num_pipe=Pipeline(steps=[
    ('imputation', SimpleImputer(strategy='median')),
    ('scaling', StandardScaler())
])
cat_pipe=Pipeline(steps=[
    ('imputation', SimpleImputer(strategy='most_frequent')),
    ('encoding', OneHotEncoder())
])

In [19]:
preprocessor=ColumnTransformer([('num_pipeline',num_pipe, num_col),
                  ('cat_pipeline', cat_pipe, cat_col)])
preprocessor

ColumnTransformer(transformers=[('num_pipeline',
                                 Pipeline(steps=[('imputation',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaling',
                                                  StandardScaler())]),
                                 ['total_bill', 'tip', 'size']),
                                ('cat_pipeline',
                                 Pipeline(steps=[('imputation',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoding',
                                                  OneHotEncoder())]),
                                 ['sex', 'smoker', 'day'])])

In [20]:
X_train=preprocessor.fit_transform(X_train)

In [21]:
X_test=preprocessor.transform(X_test)

In [22]:
X_train.shape

(195, 11)

In [23]:
X_test.shape

(49, 11)

In [24]:
# now data is ready lets build multiple models

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [25]:
models={'svc':SVC(),
       'dt classifier':DecisionTreeClassifier(),
       'logistic':LogisticRegression()}
models

{'svc': SVC(),
 'dt classifier': DecisionTreeClassifier(),
 'logistic': LogisticRegression()}

In [26]:
from sklearn.metrics import accuracy_score

def model_train_eval(X_train, X_test, y_train, y_test, models):
    evaluation={}
    for i in range(len(models)):
        model=list(models.values())[i]
        model.fit(X_train, y_train)
        y_pred=model.predict(X_test)
        score=accuracy_score(y_test, y_pred)
        evaluation[list(models.keys())[i]]=score
    return evaluation

In [27]:
model_train_eval(X_train, X_test, y_train, y_test, models)

{'svc': 0.9183673469387755,
 'dt classifier': 0.9387755102040817,
 'logistic': 0.9183673469387755}

In [40]:
from sklearn.ensemble import RandomForestClassifier
rfc=RandomForestClassifier(oob_score=True, random_state=1)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
params={
    'max_depth':[1,2,3,4,5,10,None],
    'n_estimators':[30,40,50,100,200,300],
    'criterion':['gini','entropy']
}
grid=RandomizedSearchCV(rfc, param_distributions=params, cv=5, scoring='accuracy',verbose=2, n_iter=10)
grid

RandomizedSearchCV(cv=5,
                   estimator=RandomForestClassifier(oob_score=True,
                                                    random_state=1),
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': [1, 2, 3, 4, 5, 10, None],
                                        'n_estimators': [30, 40, 50, 100, 200,
                                                         300]},
                   scoring='accuracy', verbose=2)

In [41]:
grid.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END ......criterion=gini, max_depth=5, n_estimators=200; total time=   0.5s
[CV] END ......criterion=gini, max_depth=5, n_estimators=200; total time=   0.5s
[CV] END ......criterion=gini, max_depth=5, n_estimators=200; total time=   0.5s
[CV] END ......criterion=gini, max_depth=5, n_estimators=200; total time=   0.5s
[CV] END ......criterion=gini, max_depth=5, n_estimators=200; total time=   0.5s
[CV] END ...criterion=entropy, max_depth=5, n_estimators=100; total time=   0.2s
[CV] END ...criterion=entropy, max_depth=5, n_estimators=100; total time=   0.2s
[CV] END ...criterion=entropy, max_depth=5, n_estimators=100; total time=   0.2s
[CV] END ...criterion=entropy, max_depth=5, n_estimators=100; total time=   0.3s
[CV] END ...criterion=entropy, max_depth=5, n_estimators=100; total time=   0.2s
[CV] END ....criterion=entropy, max_depth=5, n_estimators=30; total time=   0.0s
[CV] END ....criterion=entropy, max_depth=5, n_e

RandomizedSearchCV(cv=5,
                   estimator=RandomForestClassifier(oob_score=True,
                                                    random_state=1),
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': [1, 2, 3, 4, 5, 10, None],
                                        'n_estimators': [30, 40, 50, 100, 200,
                                                         300]},
                   scoring='accuracy', verbose=2)

In [42]:
grid.best_params_

{'n_estimators': 30, 'max_depth': 3, 'criterion': 'gini'}

In [43]:
grid.best_score_

np.float64(0.9794871794871796)

In [44]:
y_pred=grid.best_estimator_.predict(X_test)

In [45]:
accuracy_score(y_test, y_pred)

0.9183673469387755

In [46]:
grid.best_estimator_.oob_score_

0.9794871794871794